# COVID-19 Data Analysis and Visualization

This notebook analyzes and visualizes COVID-19 data from multiple perspectives:
- Global trends and comparisons
- Country-specific analysis
- Time series visualizations
- Statistical insights

Data source: Johns Hopkins University COVID-19 Data Repository

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("? Libraries imported successfully")

In [ ]:
# Load COVID-19 data from Johns Hopkins University repository
base_url = 'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/'

# Load confirmed, deaths, and recovered cases
confirmed_url = base_url + 'time_series_covid19_confirmed_global.csv'
deaths_url = base_url + 'time_series_covid19_deaths_global.csv'
recovered_url = base_url + 'time_series_covid19_recovered_global.csv'

print("Loading data...")
df_confirmed = pd.read_csv(confirmed_url)
df_deaths = pd.read_csv(deaths_url)

# Note: Recovered data was discontinued, we'll work with confirmed and deaths
try:
    df_recovered = pd.read_csv(recovered_url)
    print("? Recovered data loaded")
except:
    print("? Recovered data not available (discontinued by source)")
    df_recovered = None

print(f"? Confirmed cases data shape: {df_confirmed.shape}")
print(f"? Deaths data shape: {df_deaths.shape}")
print(f"\nData date range: {df_confirmed.columns[4]} to {df_confirmed.columns[-1]}")

In [ ]:
# Preview the data structure
print("=== Confirmed Cases Data Preview ===")
df_confirmed.head()

## 2. Data Preprocessing and Aggregation

In [ ]:
# Aggregate data by country
def aggregate_by_country(df):
    """Aggregate regional data by country"""
    # Group by Country/Region and sum across all provinces/states
    country_df = df.groupby('Country/Region').sum().reset_index()
    # Drop Lat and Long columns
    country_df = country_df.drop(columns=['Lat', 'Long'], errors='ignore')
    return country_df

# Create country-level dataframes
confirmed_country = aggregate_by_country(df_confirmed)
deaths_country = aggregate_by_country(df_deaths)

print(f"? Data aggregated for {len(confirmed_country)} countries/regions")
print(f"\nCountries included: {', '.join(confirmed_country['Country/Region'].head(10).values)}...")

In [ ]:
# Get latest statistics
latest_date = confirmed_country.columns[-1]
print(f"=== Latest COVID-19 Statistics (as of {latest_date}) ===")
print()

# Global totals
total_confirmed = confirmed_country[latest_date].sum()
total_deaths = deaths_country[latest_date].sum()

print(f"?? Global Total Confirmed Cases: {total_confirmed:,.0f}")
print(f"?? Global Total Deaths: {total_deaths:,.0f}")
print(f"?? Global Mortality Rate: {(total_deaths/total_confirmed*100):.2f}%")
print()

# Top 10 countries by confirmed cases
top10_confirmed = confirmed_country.nlargest(10, latest_date)[['Country/Region', latest_date]]
print("\n=== Top 10 Countries by Confirmed Cases ===")
for idx, row in top10_confirmed.iterrows():
    print(f"{row['Country/Region']:20s}: {row[latest_date]:>12,.0f}")

## 3. Global Trends Over Time

In [ ]:
# Calculate global daily totals
def calculate_global_timeseries(df):
    """Calculate global daily totals from country data"""
    date_columns = df.columns[1:]  # Skip 'Country/Region' column
    global_totals = df[date_columns].sum()
    return global_totals

global_confirmed = calculate_global_timeseries(confirmed_country)
global_deaths = calculate_global_timeseries(deaths_country)

# Convert to datetime index
dates = pd.to_datetime(global_confirmed.index)

print(f"? Time series data prepared: {len(dates)} days of data")

In [ ]:
# Plot global trends
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Cumulative cases
axes[0].plot(dates, global_confirmed.values, linewidth=2.5, color='#FF6B6B', label='Confirmed Cases')
axes[0].plot(dates, global_deaths.values, linewidth=2.5, color='#4ECDC4', label='Deaths')
axes[0].set_title('Global COVID-19 Cumulative Cases Over Time', fontsize=16, fontweight='bold', pad=20)
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Number of Cases', fontsize=12)
axes[0].legend(fontsize=11, loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].ticklabel_format(style='plain', axis='y')

# Calculate daily new cases
daily_confirmed = global_confirmed.diff().fillna(0)
daily_deaths = global_deaths.diff().fillna(0)

# 7-day rolling average for smoothing
daily_confirmed_smooth = daily_confirmed.rolling(window=7).mean()
daily_deaths_smooth = daily_deaths.rolling(window=7).mean()

# Daily new cases
axes[1].bar(dates, daily_confirmed.values, alpha=0.3, color='#FF6B6B', label='Daily Confirmed (raw)')
axes[1].plot(dates, daily_confirmed_smooth.values, linewidth=2.5, color='#FF6B6B', label='Daily Confirmed (7-day avg)')
axes[1].set_title('Global COVID-19 Daily New Cases', fontsize=16, fontweight='bold', pad=20)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Daily New Cases', fontsize=12)
axes[1].legend(fontsize=11, loc='upper left')
axes[1].grid(True, alpha=0.3)
axes[1].ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

print(f"\n?? Peak daily cases: {daily_confirmed_smooth.max():,.0f} (7-day average)")
print(f"?? Peak date: {dates[daily_confirmed_smooth.argmax()].strftime('%Y-%m-%d')}")

## 4. Country Comparisons

In [ ]:
# Select top countries for comparison
top_countries = ['US', 'India', 'Brazil', 'France', 'Germany', 'United Kingdom', 'Italy', 'Spain', 'Japan', 'South Korea']

# Filter available countries
available_countries = [c for c in top_countries if c in confirmed_country['Country/Region'].values]
print(f"Analyzing {len(available_countries)} countries: {', '.join(available_countries)}")

In [ ]:
# Plot country comparisons
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confirmed cases comparison
for country in available_countries:
    country_data = confirmed_country[confirmed_country['Country/Region'] == country]
    cases = country_data.iloc[0, 1:].values
    axes[0].plot(dates, cases, linewidth=2, label=country, marker='o', markersize=3, markevery=30)

axes[0].set_title('COVID-19 Confirmed Cases by Country', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Date', fontsize=11)
axes[0].set_ylabel('Confirmed Cases', fontsize=11)
axes[0].legend(fontsize=9, loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].ticklabel_format(style='plain', axis='y')

# Deaths comparison
for country in available_countries:
    country_data = deaths_country[deaths_country['Country/Region'] == country]
    deaths = country_data.iloc[0, 1:].values
    axes[1].plot(dates, deaths, linewidth=2, label=country, marker='o', markersize=3, markevery=30)

axes[1].set_title('COVID-19 Deaths by Country', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Date', fontsize=11)
axes[1].set_ylabel('Deaths', fontsize=11)
axes[1].legend(fontsize=9, loc='upper left')
axes[1].grid(True, alpha=0.3)
axes[1].ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

## 5. Top Countries Analysis

In [ ]:
# Top 15 countries by confirmed cases
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 by confirmed cases
top15_confirmed = confirmed_country.nlargest(15, latest_date)
countries = top15_confirmed['Country/Region'].values
cases = top15_confirmed[latest_date].values

axes[0].barh(countries, cases, color='#FF6B6B', edgecolor='black', linewidth=0.5)
axes[0].set_title('Top 15 Countries by Confirmed Cases', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Confirmed Cases', fontsize=11)
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].ticklabel_format(style='plain', axis='x')

# Add values on bars
for i, v in enumerate(cases):
    axes[0].text(v, i, f' {v:,.0f}', va='center', fontsize=9)

# Top 15 by deaths
top15_deaths = deaths_country.nlargest(15, latest_date)
countries_d = top15_deaths['Country/Region'].values
deaths_vals = top15_deaths[latest_date].values

axes[1].barh(countries_d, deaths_vals, color='#4ECDC4', edgecolor='black', linewidth=0.5)
axes[1].set_title('Top 15 Countries by Deaths', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Deaths', fontsize=11)
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].ticklabel_format(style='plain', axis='x')

# Add values on bars
for i, v in enumerate(deaths_vals):
    axes[1].text(v, i, f' {v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Mortality Rate Analysis

In [ ]:
# Calculate mortality rates for countries with significant cases
min_cases = 100000  # Only consider countries with at least 100k cases

mortality_data = []
for idx, row in confirmed_country.iterrows():
    country = row['Country/Region']
    confirmed = row[latest_date]
    
    if confirmed >= min_cases:
        deaths_row = deaths_country[deaths_country['Country/Region'] == country]
        if not deaths_row.empty:
            deaths = deaths_row[latest_date].values[0]
            mortality_rate = (deaths / confirmed) * 100
            mortality_data.append({
                'Country': country,
                'Confirmed': confirmed,
                'Deaths': deaths,
                'Mortality Rate (%)': mortality_rate
            })

mortality_df = pd.DataFrame(mortality_data)
mortality_df = mortality_df.sort_values('Mortality Rate (%)', ascending=False)

print(f"\n=== Mortality Rate Analysis (Countries with 100k+ cases) ===")
print(f"\nTotal countries analyzed: {len(mortality_df)}")
print(f"\nTop 10 Highest Mortality Rates:")
print(mortality_df.head(10).to_string(index=False))
print(f"\nTop 10 Lowest Mortality Rates:")
print(mortality_df.tail(10).to_string(index=False))

In [ ]:
# Visualize mortality rates
fig, ax = plt.subplots(figsize=(14, 8))

# Top 20 countries by mortality rate
top20_mortality = mortality_df.head(20)
countries = top20_mortality['Country'].values
rates = top20_mortality['Mortality Rate (%)'].values

bars = ax.barh(countries, rates, color='#E74C3C', edgecolor='black', linewidth=0.5)
ax.set_title('Top 20 Countries by COVID-19 Mortality Rate\n(Countries with 100k+ cases)', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Mortality Rate (%)', fontsize=11)
ax.set_ylabel('Country', fontsize=11)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Add values on bars
for i, v in enumerate(rates):
    ax.text(v, i, f' {v:.2f}%', va='center', fontsize=9)

# Add global average line
global_mortality = (total_deaths / total_confirmed) * 100
ax.axvline(global_mortality, color='blue', linestyle='--', linewidth=2, label=f'Global Average: {global_mortality:.2f}%')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 7. Growth Rate Analysis

In [ ]:
# Calculate 7-day growth rate for selected countries
def calculate_growth_rate(country_df, country_name, days=7):
    """Calculate percentage growth over specified days"""
    country_data = country_df[country_df['Country/Region'] == country_name]
    if country_data.empty:
        return None
    
    date_cols = country_data.columns[1:]
    current_cases = country_data[date_cols[-1]].values[0]
    past_cases = country_data[date_cols[-days-1]].values[0]
    
    if past_cases == 0:
        return 0
    
    growth_rate = ((current_cases - past_cases) / past_cases) * 100
    return growth_rate

# Calculate growth rates for top countries
growth_data = []
for country in available_countries:
    growth_7d = calculate_growth_rate(confirmed_country, country, days=7)
    growth_30d = calculate_growth_rate(confirmed_country, country, days=30)
    
    if growth_7d is not None:
        growth_data.append({
            'Country': country,
            '7-Day Growth (%)': growth_7d,
            '30-Day Growth (%)': growth_30d
        })

growth_df = pd.DataFrame(growth_data)
print("=== Recent Growth Rates ===")
print(growth_df.to_string(index=False))

In [ ]:
# Visualize growth rates
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(growth_df))
width = 0.35

bars1 = ax.bar(x - width/2, growth_df['7-Day Growth (%)'], width, label='7-Day Growth', color='#3498DB')
bars2 = ax.bar(x + width/2, growth_df['30-Day Growth (%)'], width, label='30-Day Growth', color='#E67E22')

ax.set_title('COVID-19 Growth Rates by Country', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Growth Rate (%)', fontsize=11)
ax.set_xlabel('Country', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(growth_df['Country'], rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 8. Heatmap of Cases Over Time

In [ ]:
# Create heatmap for top 15 countries
top15 = confirmed_country.nlargest(15, latest_date)
countries_list = top15['Country/Region'].values

# Get data for last 180 days (approximately 6 months)
date_cols = confirmed_country.columns[1:]
last_180_cols = date_cols[-180:]

# Create matrix for heatmap
heatmap_data = []
for country in countries_list:
    country_data = confirmed_country[confirmed_country['Country/Region'] == country][last_180_cols]
    heatmap_data.append(country_data.values[0])

heatmap_matrix = np.array(heatmap_data)

# Plot heatmap
fig, ax = plt.subplots(figsize=(16, 8))

# Use log scale for better visualization
heatmap_matrix_log = np.log10(heatmap_matrix + 1)  # +1 to avoid log(0)

im = ax.imshow(heatmap_matrix_log, cmap='YlOrRd', aspect='auto')
ax.set_yticks(np.arange(len(countries_list)))
ax.set_yticklabels(countries_list)
ax.set_xlabel('Days (Last 180 days)', fontsize=11)
ax.set_title('COVID-19 Cases Heatmap - Top 15 Countries (Log Scale)', fontsize=14, fontweight='bold', pad=15)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Log10(Cases + 1)', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Summary Statistics and Insights

In [ ]:
# Generate comprehensive summary
print("="*70)
print("COVID-19 DATA ANALYSIS SUMMARY")
print("="*70)
print(f"\nData as of: {latest_date}")
print(f"Total countries/regions tracked: {len(confirmed_country)}")
print()
print("?? GLOBAL STATISTICS")
print("-" * 70)
print(f"Total Confirmed Cases: {total_confirmed:>20,.0f}")
print(f"Total Deaths: {total_deaths:>20,.0f}")
print(f"Global Mortality Rate: {(total_deaths/total_confirmed*100):>19.2f}%")
print()

# Most affected country
most_cases_country = confirmed_country.loc[confirmed_country[latest_date].idxmax(), 'Country/Region']
most_cases_value = confirmed_country[latest_date].max()
print(f"Most Affected Country: {most_cases_country} ({most_cases_value:,.0f} cases)")
print()

print("?? RECENT TRENDS (Last 7 Days)")
print("-" * 70)
week_ago_col = date_cols[-8]
new_cases_week = total_confirmed - confirmed_country[week_ago_col].sum()
new_deaths_week = total_deaths - deaths_country[week_ago_col].sum()
print(f"New Cases (7 days): {new_cases_week:>20,.0f}")
print(f"New Deaths (7 days): {new_deaths_week:>19,.0f}")
print(f"Average Daily New Cases: {(new_cases_week/7):>16,.0f}")
print()

print("?? TOP 5 COUNTRIES BY CONFIRMED CASES")
print("-" * 70)
top5 = confirmed_country.nlargest(5, latest_date)
for i, (idx, row) in enumerate(top5.iterrows(), 1):
    country = row['Country/Region']
    cases = row[latest_date]
    deaths_val = deaths_country[deaths_country['Country/Region'] == country][latest_date].values[0]
    mort_rate = (deaths_val / cases) * 100
    print(f"{i}. {country:20s} - {cases:>12,.0f} cases (Mortality: {mort_rate:.2f}%)")

print("\n" + "="*70)

## 10. Key Insights and Conclusions

Based on the analysis above, we can observe:

1. **Global Impact**: The pandemic has affected millions worldwide with varying mortality rates across countries

2. **Regional Differences**: Different countries show different patterns based on:
   - Healthcare infrastructure
   - Population density
   - Public health measures
   - Vaccination rates

3. **Trends**: The time series analysis shows multiple waves of infections with varying intensity

4. **Mortality Rates**: Vary significantly by country, influenced by:
   - Age demographics
   - Healthcare capacity
   - Testing rates
   - Comorbidities

5. **Data Limitations**: 
   - Testing rates vary by country
   - Reporting standards differ
   - Asymptomatic cases often unreported

## 11. Export Results (Optional)

In [ ]:
# Export summary statistics to CSV
summary_data = []
for idx, row in confirmed_country.iterrows():
    country = row['Country/Region']
    confirmed = row[latest_date]
    deaths_row = deaths_country[deaths_country['Country/Region'] == country]
    
    if not deaths_row.empty:
        deaths = deaths_row[latest_date].values[0]
        mortality_rate = (deaths / confirmed * 100) if confirmed > 0 else 0
        
        summary_data.append({
            'Country': country,
            'Confirmed_Cases': confirmed,
            'Deaths': deaths,
            'Mortality_Rate_Percent': round(mortality_rate, 2)
        })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Confirmed_Cases', ascending=False)

# Save to CSV
output_file = 'covid19_summary.csv'
summary_df.to_csv(output_file, index=False)
print(f"? Summary statistics exported to: {output_file}")
print(f"  Total records: {len(summary_df)}")
print(f"\nFirst few rows:")
summary_df.head(10)